# RT-DETR Model Code

In [1]:
import torchvision.datasets as datasets

import os
import torch
import torch.nn as nn
import torchvision.transforms.v2
from torch.utils.data.dataset import Dataset
import xml.etree.ElementTree as ET
from torchvision import tv_tensors
from torchvision.io import read_image
import yaml
from tqdm import tqdm
from torch.utils.data.dataloader import DataLoader
from torch.utils.data import random_split
from torch.optim.lr_scheduler import MultiStepLR
import torchvision.models
from torchvision.models import resnet34
from scipy.optimize import linear_sum_assignment
from collections import defaultdict
import numpy as np
import random

In [3]:
path = "mor_cyclic_voc_results/best_mor_cyclic_detr.pth"
checkpoint = torch.load(path, map_location=torch.device('cpu'), weights_only=False)

if isinstance(checkpoint, dict) and 'model_state_dict' in checkpoint:
    print(checkpoint.keys())

# # Bare state_dict from an older run.
# model.load_state_dict(checkpoint)
# print('Loaded a bare state_dict checkpoint; optimizer/scheduler state and '
#         'epoch counter are not available, training would restart from epoch 0.')
# return 0, 0

dict_keys(['epoch', 'model_state_dict', 'optimizer_state_dict', 'scheduler_state_dict', 'loss', 'steps', 'min_val_loss', 'train_cls_loss', 'train_loc_loss', 'val_cls_loss', 'val_loc_loss'])


In [4]:
checkpoint['epoch'], checkpoint['loss'], checkpoint['min_val_loss'], checkpoint['val_cls_loss'], checkpoint['val_loc_loss']

(150,
 8.410957336425781,
 9.485021392504374,
 3.2938641536803472,
 6.191157238824027)

In [ ]:
print(torch.cuda.is_available())

print(os.environ.get('CUDA_VISIBLE_DEVICES'))

print(torch.__version__)

print(torch.version.cuda)

In [ ]:
def load_images_and_anns(im_sets, label2idx, ann_fname, split):
    r"""
    Method to get the xml files and for each file
    get all the objects and their ground truth detection
    information for the dataset
    :param im_sets: Sets of images to consider
    :param label2idx: Class Name to index mapping for dataset
    :param ann_fname: txt file containing image names{trainval.txt/test.txt}
    :param split: train/test
    :return:
    """
    im_infos = []
    for im_set in im_sets:
        im_names = []
        # Fetch all image names in txt file for this imageset
        for line in open(os.path.join(
                im_set, 'ImageSets', 'Main', '{}.txt'.format(ann_fname))):
            im_names.append(line.strip())

        # Set annotation and image path
        ann_dir = os.path.join(im_set, 'Annotations')
        im_dir = os.path.join(im_set, 'JPEGImages')

        for im_name in im_names:
            ann_file = os.path.join(ann_dir, '{}.xml'.format(im_name))
            im_info = {}
            ann_info = ET.parse(ann_file)
            root = ann_info.getroot()
            size = root.find('size')
            width = int(size.find('width').text)
            height = int(size.find('height').text)
            im_info['img_id'] = os.path.basename(ann_file).split('.xml')[0]
            im_info['filename'] = os.path.join(
                im_dir, '{}.jpg'.format(im_info['img_id'])
            )
            im_info['width'] = width
            im_info['height'] = height
            detections = []
            for obj in ann_info.findall('object'):
                det = {}
                label = label2idx[obj.find('name').text]
                difficult = int(obj.find('difficult').text)
                bbox_info = obj.find('bndbox')
                bbox = [
                    int(bbox_info.find('xmin').text) - 1,
                    int(bbox_info.find('ymin').text) - 1,
                    int(bbox_info.find('xmax').text) - 1,
                    int(bbox_info.find('ymax').text) - 1
                ]
                det['label'] = label
                det['bbox'] = bbox
                det['difficult'] = difficult
                detections.append(det)
            im_info['detections'] = detections
            # Because we are using 25 as num_queries,
            # so we ignore all images in VOC with greater
            # than 25 target objects.
            # This is okay, since this just means we are
            # ignoring a small number of images(15 to be precise)
            if len(detections) <= 25:
                im_infos.append(im_info)
    print('Total {} images found'.format(len(im_infos)))
    return im_infos

In [ ]:
class VOCDataset(Dataset):
    def __init__(self, split, im_sets, im_size=640):
        self.split = split

        # Imagesets for this dataset instance (VOC2007/VOC2007+VOC2012/VOC2007-test)
        self.im_sets = im_sets
        self.fname = 'trainval' if self.split == 'train' else 'test'
        self.im_size = im_size
        self.im_mean = [123.0, 117.0, 104.0]
        self.imagenet_mean = [0.485, 0.456, 0.406]
        self.imagenet_std = [0.229, 0.224, 0.225]

        # Train and test transformations
        self.transforms = {
            'train': torchvision.transforms.v2.Compose([
                torchvision.transforms.v2.RandomHorizontalFlip(p=0.5),
                torchvision.transforms.v2.RandomZoomOut(fill=self.im_mean),
                torchvision.transforms.v2.RandomIoUCrop(),
                torchvision.transforms.v2.RandomPhotometricDistort(),
                torchvision.transforms.v2.Resize(size=(self.im_size, self.im_size)),
                torchvision.transforms.v2.SanitizeBoundingBoxes(
                    labels_getter=lambda transform_input:
                    transform_input[1]["labels"]),
                # torchvision.transforms.v2.SanitizeBoundingBoxes(
                #     labels_getter=lambda transform_input:
                #     (transform_input[1]["labels"], transform_input[1]["difficult"])),
                torchvision.transforms.v2.ToPureTensor(),
                torchvision.transforms.v2.ToDtype(torch.float32, scale=True),
                torchvision.transforms.v2.Normalize(mean=self.imagenet_mean,
                                                    std=self.imagenet_std)

            ]),
            'test': torchvision.transforms.v2.Compose([
                torchvision.transforms.v2.Resize(size=(self.im_size, self.im_size)),
                torchvision.transforms.v2.ToPureTensor(),
                torchvision.transforms.v2.ToDtype(torch.float32, scale=True),
                torchvision.transforms.v2.Normalize(mean=self.imagenet_mean,
                                                    std=self.imagenet_std)
            ]),
        }

        classes = [
            'person', 'bird', 'cat', 'cow', 'dog', 'horse', 'sheep',
            'aeroplane', 'bicycle', 'boat', 'bus', 'car', 'motorbike', 'train',
            'bottle', 'chair', 'diningtable', 'pottedplant', 'sofa', 'tvmonitor'
        ]
        classes = sorted(classes)
        # We need to add background class as well with 0 index
        classes = ['background'] + classes

        self.label2idx = {classes[idx]: idx for idx in range(len(classes))}
        self.idx2label = {idx: classes[idx] for idx in range(len(classes))}
        print(self.idx2label)
        self.images_info = load_images_and_anns(self.im_sets,
                                                self.label2idx,
                                                self.fname,
                                                self.split)

    def __len__(self):
        return len(self.images_info)

    def __getitem__(self, index):
        im_info = self.images_info[index]
        im = read_image(im_info['filename'])

        # Get annotations for this image
        targets = {}
        targets['boxes'] = tv_tensors.BoundingBoxes(
            [detection['bbox'] for detection in im_info['detections']],
            format='XYXY', canvas_size=im.shape[-2:])
        targets['labels'] = torch.as_tensor(
            [detection['label'] for detection in im_info['detections']])
        targets['difficult'] = torch.as_tensor(
            [detection['difficult']for detection in im_info['detections']])

        # Transform the image and targets
        transformed_info = self.transforms[self.split](im, targets)
        im_tensor, targets = transformed_info

        h, w = im_tensor.shape[-2:]

        # Boxes returned are in x1y1x2y2 format normalized from 0-1
        wh_tensor = torch.as_tensor([[w, h, w, h]]).expand_as(targets['boxes'])
        targets['boxes'] = targets['boxes'] / wh_tensor
        return im_tensor, targets, im_info['filename']

## Deformable attention, anchors and shared utilities

`MultiScaleDeformableAttention` is **not** part of torchvision in any release -
it lives in `mmcv.ops` and in the original Deformable-DETR CUDA extension. The
cell below therefore owns the module and picks its sampling kernel at import:

| environment | backend |
|---|---|
| Windows / CPU / anything | pure-PyTorch `F.grid_sample` (default, always works) |
| Kaggle or Linux with `pip install mmcv` | fused CUDA kernel, picked up automatically |

Selection is automatic and printed on import - the same notebook runs unchanged
in both places and simply goes faster where the kernel is available.

In [ ]:
# ---------------------------------------------------------------------------
# Shared utilities + multi-scale deformable attention.
#
# `MultiScaleDeformableAttention` is NOT part of torchvision (any version) -- it
# lives in mmcv.ops / the original Deformable-DETR CUDA extension. So we own the
# module here and only swap the inner sampling kernel:
#
#   * default: a pure-PyTorch F.grid_sample core. Runs everywhere (Windows CPU,
#     Windows CUDA, Kaggle/Linux). ~2-3x slower than the fused kernel.
#   * fast path: if mmcv or the Deformable-DETR CUDA extension happens to be
#     importable AND the tensors are on CUDA, the fused kernel is used instead.
#     Both expose the identical signature
#         fn(value, spatial_shapes, level_start_index,
#            sampling_locations, attention_weights, im2col_step)
#     so the module, its parameters and its checkpoints are backend-independent.
#
# To get the fast path on Kaggle:  pip install mmcv
# (the 2021 `MultiScaleDeformableAttention` PyPI wheel is ABI-stale against
#  torch 2.x -- prefer mmcv or a fresh build of the Deformable-DETR op.)
# ---------------------------------------------------------------------------
import math

import torch
import torch.nn as nn
import torch.nn.functional as F

_MSDA_CUDA_FN = None
_MSDA_BACKEND = 'pytorch (grid_sample)'
try:
    from mmcv.ops.multi_scale_deform_attn import (
        MultiScaleDeformableAttnFunction as _MSDA_CUDA_FN,
    )
    _MSDA_BACKEND = 'mmcv fused CUDA kernel'
except Exception:
    try:
        from MultiScaleDeformableAttention import (
            MSDeformAttnFunction as _MSDA_CUDA_FN,
        )
        _MSDA_BACKEND = 'Deformable-DETR fused CUDA kernel'
    except Exception:
        pass
print('MSDeformAttn backend: {}'.format(_MSDA_BACKEND))


def inverse_sigmoid(x, eps=1e-5):
    r"""
    Logit of x, with clamping so the log never sees 0 or a negative.

    :param x: tensor expected to lie in [0, 1]
    :param eps: floor applied to both x and 1-x
    :return: tensor of the same shape in unbounded space
    """
    x = x.clamp(min=0., max=1.)
    return torch.log(x.clamp(min=eps) / (1. - x).clamp(min=eps))


def get_spatial_position_embedding(pos_emb_dim, feat_map):
    r"""
    2D sine/cosine position embedding, one vector per feature-map cell.

    Half the dimensions encode the row index and half the column index, each
    split again into sin and cos - hence the divisible-by-4 requirement.

    :param pos_emb_dim: embedding dimension (must be divisible by 4)
    :param feat_map: (B, C, H, W) tensor; only its shape and device are used
    :return: (H * W, pos_emb_dim) tensor
    """
    assert pos_emb_dim % 4 == 0, ('Position embedding dimension '
                                  'must be divisible by 4')
    grid_size_h, grid_size_w = feat_map.shape[2], feat_map.shape[3]
    grid_h = torch.arange(grid_size_h,
                          dtype=torch.float32,
                          device=feat_map.device)
    grid_w = torch.arange(grid_size_w,
                          dtype=torch.float32,
                          device=feat_map.device)
    grid = torch.meshgrid(grid_h, grid_w, indexing='ij')
    grid = torch.stack(grid, dim=0)

    # grid_h_positions -> (Number of grid cell tokens,)
    grid_h_positions = grid[0].reshape(-1)
    grid_w_positions = grid[1].reshape(-1)

    # factor = 10000^(2i/d_model)
    factor = 10000 ** ((torch.arange(
        start=0,
        end=pos_emb_dim // 4,
        dtype=torch.float32,
        device=feat_map.device) / (pos_emb_dim // 4))
    )

    grid_h_emb = grid_h_positions[:, None].repeat(1, pos_emb_dim // 4) / factor
    grid_h_emb = torch.cat([
        torch.sin(grid_h_emb),
        torch.cos(grid_h_emb)
    ], dim=-1)
    # grid_h_emb -> (Number of grid cell tokens, pos_emb_dim // 2)

    grid_w_emb = grid_w_positions[:, None].repeat(1, pos_emb_dim // 4) / factor
    grid_w_emb = torch.cat([
        torch.sin(grid_w_emb),
        torch.cos(grid_w_emb)
    ], dim=-1)
    pos_emb = torch.cat([grid_h_emb, grid_w_emb], dim=-1)

    # pos_emb -> (Number of grid cell tokens, pos_emb_dim)
    return pos_emb


def generate_anchors(spatial_shapes, grid_size=0.05, eps=1e-2,
                     dtype=torch.float32, device='cpu'):
    r"""
    RT-DETR per-level anchor grid, used to turn the encoder bbox head into a
    *delta* predictor instead of an absolute one.

    One anchor per feature-map cell: centre at the cell centre, width/height a
    fixed fraction of the image that doubles with every level. Returned in
    inverse-sigmoid space so the head can simply add its delta before the
    sigmoid. Anchors whose centre or size falls outside (eps, 1-eps) are marked
    invalid and pushed to a large value, so sigmoid saturates them out.

    :param spatial_shapes: (num_levels, 2) tensor of (H, W) per level
    :param grid_size: base anchor side length at level 0, as a fraction of the image
    :param eps: validity margin
    :return: (anchors, valid_mask) of shapes (1, sum(H*W), 4) and (1, sum(H*W), 1)
    """
    anchors = []
    for level, (h, w) in enumerate(spatial_shapes.tolist()):
        grid_y, grid_x = torch.meshgrid(
            torch.arange(h, dtype=dtype, device=device),
            torch.arange(w, dtype=dtype, device=device),
            indexing='ij')
        # Cell centres, normalised to [0, 1]
        grid_xy = torch.stack([grid_x, grid_y], dim=-1)
        grid_xy = (grid_xy + 0.5) / torch.tensor([w, h], dtype=dtype,
                                                 device=device)
        wh = torch.ones_like(grid_xy) * grid_size * (2.0 ** level)
        anchors.append(torch.cat([grid_xy, wh], dim=-1).reshape(-1, 4))

    anchors = torch.cat(anchors, dim=0).unsqueeze(0)
    # anchors -> (1, sum(H*W), 4) in cxcywh, [0, 1]

    valid_mask = ((anchors > eps) & (anchors < 1 - eps)).all(dim=-1,
                                                             keepdim=True)
    # valid_mask -> (1, sum(H*W), 1)

    anchors = inverse_sigmoid(anchors)
    # Large-but-finite instead of inf: sigmoid still saturates to 1 and the
    # gradient is 0, but no inf ever enters the arithmetic.
    anchors = torch.where(valid_mask, anchors,
                          torch.full_like(anchors, 1e4))
    return anchors, valid_mask


def ms_deform_attn_core_pytorch(value, value_spatial_shapes,
                                sampling_locations, attention_weights):
    r"""
    Portable reference implementation of multi-scale deformable attention.

    Each level is grid_sample'd independently at the predicted sampling
    locations, then all (level, point) samples are combined by the predicted
    attention weights. Numerically equivalent to the fused CUDA kernel.

    :param value: (B, sum(H*W), num_heads, head_dim)
    :param value_spatial_shapes: (num_levels, 2) tensor of (H, W)
    :param sampling_locations: (B, num_queries, num_heads, num_levels, num_points, 2)
        in [0, 1] normalised (x, y)
    :param attention_weights: (B, num_queries, num_heads, num_levels, num_points)
    :return: (B, num_queries, num_heads * head_dim)
    """
    batch_size, _, num_heads, head_dim = value.shape
    _, num_queries, _, num_levels, num_points, _ = sampling_locations.shape

    shapes = value_spatial_shapes.tolist()
    value_list = value.split([h * w for h, w in shapes], dim=1)

    # grid_sample wants [-1, 1] coordinates
    sampling_grids = 2 * sampling_locations - 1

    sampling_value_list = []
    for level, (h, w) in enumerate(shapes):
        # (B, H*W, heads, head_dim) -> (B*heads, head_dim, H, W)
        value_l = (value_list[level]
                   .flatten(2)
                   .transpose(1, 2)
                   .reshape(batch_size * num_heads, head_dim, h, w))
        # (B, queries, heads, points, 2) -> (B*heads, queries, points, 2)
        grid_l = (sampling_grids[:, :, :, level]
                  .transpose(1, 2)
                  .flatten(0, 1))
        sampling_value_list.append(
            F.grid_sample(value_l, grid_l,
                          mode='bilinear',
                          padding_mode='zeros',
                          align_corners=False)
        )
        # each -> (B*heads, head_dim, queries, points)

    attention_weights = (attention_weights
                         .transpose(1, 2)
                         .reshape(batch_size * num_heads, 1, num_queries,
                                  num_levels * num_points))
    output = (torch.stack(sampling_value_list, dim=-2).flatten(-2)
              * attention_weights).sum(-1)
    # output -> (B*heads, head_dim, queries)
    output = output.view(batch_size, num_heads * head_dim, num_queries)
    return output.transpose(1, 2).contiguous()


class MSDeformAttn(nn.Module):
    r"""
    Multi-scale deformable attention (Deformable-DETR / RT-DETR).

    Instead of attending to every one of the ~8400 encoder tokens, each query
    predicts ``num_points`` sampling offsets per head per level around its
    reference point and attends only to those. Cost is independent of the
    feature-map size, which is what makes the decoder real-time.

    :param d_model: model dimension
    :param num_heads: attention heads
    :param num_levels: number of feature-map levels in the flattened memory
    :param num_points: sampling points per head, per level
    """

    def __init__(self, d_model=256, num_heads=8, num_levels=3, num_points=4):
        super().__init__()
        assert d_model % num_heads == 0, 'd_model must be divisible by num_heads'
        self.d_model = d_model
        self.num_heads = num_heads
        self.num_levels = num_levels
        self.num_points = num_points
        self.head_dim = d_model // num_heads
        self.im2col_step = 64

        self.sampling_offsets = nn.Linear(
            d_model, num_heads * num_levels * num_points * 2)
        self.attention_weights = nn.Linear(
            d_model, num_heads * num_levels * num_points)
        self.value_proj = nn.Linear(d_model, d_model)
        self.output_proj = nn.Linear(d_model, d_model)

        self._reset_parameters()

    def _reset_parameters(self):
        r"""
        The published init, which deformable attention genuinely needs: offsets
        start as a ring of ``num_heads`` directions (growing outwards with the
        point index) so the heads immediately look in different directions, and
        the attention weights start uniform.
        """
        nn.init.constant_(self.sampling_offsets.weight, 0.)
        thetas = (torch.arange(self.num_heads, dtype=torch.float32)
                  * (2.0 * math.pi / self.num_heads))
        grid_init = torch.stack([thetas.cos(), thetas.sin()], dim=-1)
        grid_init = grid_init / grid_init.abs().max(dim=-1, keepdim=True)[0]
        grid_init = (grid_init.view(self.num_heads, 1, 1, 2)
                     .repeat(1, self.num_levels, self.num_points, 1))
        for point in range(self.num_points):
            grid_init[:, :, point, :] *= point + 1
        with torch.no_grad():
            self.sampling_offsets.bias.copy_(grid_init.view(-1))

        nn.init.constant_(self.attention_weights.weight, 0.)
        nn.init.constant_(self.attention_weights.bias, 0.)
        nn.init.xavier_uniform_(self.value_proj.weight)
        nn.init.constant_(self.value_proj.bias, 0.)
        nn.init.xavier_uniform_(self.output_proj.weight)
        nn.init.constant_(self.output_proj.bias, 0.)

    def forward(self, query, reference_points, input_flatten,
                input_spatial_shapes, input_level_start_index):
        r"""
        :param query: (B, num_queries, d_model)
        :param reference_points: (B, num_queries, num_levels, 2 or 4),
            normalised to [0, 1]. 2 -> (cx, cy); 4 -> (cx, cy, w, h), where the
            w/h additionally scale the sampling offsets.
        :param input_flatten: (B, sum(H*W), d_model) flattened multi-scale memory
        :param input_spatial_shapes: (num_levels, 2) tensor of (H, W)
        :param input_level_start_index: (num_levels,) offsets into input_flatten
        :return: (B, num_queries, d_model)
        """
        batch_size, num_queries, _ = query.shape
        _, num_value, _ = input_flatten.shape
        assert int(input_spatial_shapes.prod(dim=1).sum()) == num_value, (
            'spatial_shapes do not account for every token in the memory')

        value = self.value_proj(input_flatten)
        value = value.view(batch_size, num_value, self.num_heads, self.head_dim)

        sampling_offsets = self.sampling_offsets(query).view(
            batch_size, num_queries, self.num_heads,
            self.num_levels, self.num_points, 2)

        attention_weights = self.attention_weights(query).view(
            batch_size, num_queries, self.num_heads,
            self.num_levels * self.num_points)
        attention_weights = attention_weights.softmax(-1).view(
            batch_size, num_queries, self.num_heads,
            self.num_levels, self.num_points)

        if reference_points.shape[-1] == 2:
            # Offsets are in units of one feature-map cell of their own level
            offset_normalizer = torch.stack(
                [input_spatial_shapes[..., 1], input_spatial_shapes[..., 0]],
                dim=-1)
            sampling_locations = (
                reference_points[:, :, None, :, None, :]
                + sampling_offsets
                / offset_normalizer[None, None, None, :, None, :])
        elif reference_points.shape[-1] == 4:
            # Offsets are in units of half the reference box own size
            sampling_locations = (
                reference_points[:, :, None, :, None, :2]
                + sampling_offsets / self.num_points
                * reference_points[:, :, None, :, None, 2:] * 0.5)
        else:
            raise ValueError(
                'reference_points last dim must be 2 or 4, got {}'.format(
                    reference_points.shape[-1]))

        if _MSDA_CUDA_FN is not None and value.is_cuda:
            output = _MSDA_CUDA_FN.apply(
                value, input_spatial_shapes, input_level_start_index,
                sampling_locations, attention_weights, self.im2col_step)
        else:
            output = ms_deform_attn_core_pytorch(
                value, input_spatial_shapes,
                sampling_locations, attention_weights)

        return self.output_proj(output)

In [ ]:
class TransformerEncoder(nn.Module):
    r"""
    Encoder for transformer of DETR.
    This has sequence of encoder layers.
    Each layer has the following modules:
        1. LayerNorm for Self Attention
        2. Self Attention
        3. LayerNorm for MLP
        4. MLP
    """
    def __init__(self, num_layers, num_heads, d_model, ff_inner_dim,
                 dropout_prob=0.0):
        super().__init__()
        self.num_layers = num_layers
        self.dropout_prob = dropout_prob

        # Self Attention Module for all encoder layers
        self.attns = nn.ModuleList(
                [
                    nn.MultiheadAttention(d_model, num_heads,
                                          dropout=self.dropout_prob,
                                          batch_first=True)
                    for _ in range(num_layers)
                ]
            )

        # MLP Module for all encoder layers
        self.ffs = nn.ModuleList(
            [
                nn.Sequential(
                    nn.Linear(d_model, ff_inner_dim),
                    nn.ReLU(),
                    nn.Dropout(self.dropout_prob),
                    nn.Linear(ff_inner_dim, d_model),
                )
                for _ in range(num_layers)
            ])

        # Norm for Self Attention for all encoder layers
        self.attn_norms = nn.ModuleList(
                [
                    nn.LayerNorm(d_model)
                    for _ in range(num_layers)
                ])

        # Norm for MLP for all encoder layers
        self.ff_norms = nn.ModuleList(
            [
                nn.LayerNorm(d_model)
                for _ in range(num_layers)
            ])

        # Dropout for Self Attention for all encoder layers
        self.attn_dropouts = nn.ModuleList(
            [
                nn.Dropout(self.dropout_prob)
                for _ in range(num_layers)
            ])

        # Dropout for MLP for all encoder layers
        self.ff_dropouts = nn.ModuleList(
            [
                nn.Dropout(self.dropout_prob)
                for _ in range(num_layers)
            ])

        # Norm for encoder output
        self.output_norm = nn.LayerNorm(d_model)

    def forward(self, x, spatial_position_embedding):
        out = x
        attn_weights = []
        for i in range(self.num_layers):
            # Norm, Self Attention, Dropout and Residual
            in_attn = self.attn_norms[i](out)
            # Add spatial position embedding
            # to q,k for self attention
            q = in_attn + spatial_position_embedding
            k = in_attn + spatial_position_embedding
            out_attn, attn_weight = self.attns[i](
                query=q,
                key=k,
                value=in_attn
            )
            attn_weights.append(attn_weight)
            out_attn = self.attn_dropouts[i](out_attn)
            out = out + out_attn

            # Norm, MLP, Dropout and Residual
            in_ff = self.ff_norms[i](out)
            out_ff = self.ffs[i](in_ff)
            out_ff = self.ff_dropouts[i](out_ff)
            out = out + out_ff

        # Output Normalization
        out = self.output_norm(out)
        return out, torch.stack(attn_weights)


In [ ]:
class DeformableCrossAttnBlock(nn.Module):
    r"""
    Thin wrapper around :class:`MSDeformAttn` that broadcasts a single reference
    box per query across all feature levels and applies output dropout.

    :param d_model: model dimension
    :param num_heads: attention heads
    :param dropout: dropout applied to the attention output
    :param num_levels: number of feature-map levels in the flattened memory
    :param num_points: sampling points per head, per level
    """

    def __init__(self, d_model, num_heads, dropout=0.0,
                 num_levels=3, num_points=4):
        super().__init__()
        self.d_model = d_model
        self.num_heads = num_heads
        self.num_levels = num_levels
        self.num_points = num_points

        self.ms_deform_attn = MSDeformAttn(
            d_model=d_model,
            num_heads=num_heads,
            num_levels=num_levels,
            num_points=num_points
        )
        self.dropout = nn.Dropout(dropout)

    def forward(self, query, reference_points, input_flatten,
                input_spatial_shapes, input_level_start_index):
        r"""
        :param query: (B, num_queries, d_model)
        :param reference_points: (B, num_queries, 4) cxcywh in [0, 1]
        :param input_flatten: (B, sum(H*W), d_model)
        :param input_spatial_shapes: (num_levels, 2) tensor of (H, W)
        :param input_level_start_index: (num_levels,) offsets into input_flatten
        :return: (B, num_queries, d_model)
        """
        # Deformable attention wants one reference point *per level*. Every
        # image is resized to a fixed square with no padding, so all valid
        # ratios are 1 and the same box can simply be reused at every level.
        reference_points = (reference_points.unsqueeze(2)
                            .expand(-1, -1, self.num_levels, -1))
        # reference_points -> (B, num_queries, num_levels, 4)

        output = self.ms_deform_attn(
            query=query,
            reference_points=reference_points,
            input_flatten=input_flatten,
            input_spatial_shapes=input_spatial_shapes,
            input_level_start_index=input_level_start_index
        )
        return self.dropout(output)


class DeformableTransformerDecoder(nn.Module):
    r"""
    Decoder for the deformable transformer of RT-DETR.
    This has a sequence of decoder layers.
    Each layer has the following modules:
        1. LayerNorm for Self Attention
        2. Self Attention
        3. LayerNorm for Deformable Cross Attention on the encoder output
        4. Deformable Cross Attention
        5. LayerNorm for MLP
        6. MLP
        7. A bbox head predicting a refinement of the reference box

    Every layer returns its own output and its own refined boxes, so the caller
    can apply deep supervision across all of them.
    """

    def __init__(self, num_layers, num_heads, d_model, ff_inner_dim,
                 dropout_prob=0.0, num_levels=3, num_points=4):
        super().__init__()
        self.num_layers = num_layers
        self.d_model = d_model
        self.dropout_prob = dropout_prob
        self.num_levels = num_levels

        # Self Attention module for all decoder layers
        self.attns = nn.ModuleList(
            [
                nn.MultiheadAttention(d_model, num_heads,
                                      dropout=self.dropout_prob,
                                      batch_first=True)
                for _ in range(num_layers)
            ])

        # Cross Attention Module for all decoder layers
        self.cross_attns = nn.ModuleList([
            DeformableCrossAttnBlock(d_model, num_heads,
                                     dropout=self.dropout_prob,
                                     num_levels=num_levels,
                                     num_points=num_points)
            for _ in range(num_layers)
        ])

        # MLP Module for all decoder layers
        self.ffs = nn.ModuleList(
            [
                nn.Sequential(
                    nn.Linear(d_model, ff_inner_dim),
                    nn.ReLU(),
                    nn.Dropout(self.dropout_prob),
                    nn.Linear(ff_inner_dim, d_model),
                )
                for _ in range(num_layers)
            ])

        # Norm for Self Attention Module for all decoder layers
        self.attn_norms = nn.ModuleList(
            [
                nn.LayerNorm(d_model)
                for _ in range(num_layers)
            ])

        # Norm for Cross Attention Module for all decoder layers
        self.cross_attn_norms = nn.ModuleList(
            [
                nn.LayerNorm(d_model)
                for _ in range(num_layers)
            ])

        # Norm for MLP Module for all decoder layers
        self.ff_norms = nn.ModuleList(
            [
                nn.LayerNorm(d_model)
                for _ in range(num_layers)
            ])

        # Dropout for Attention Module for all decoder layers
        self.attn_dropouts = nn.ModuleList(
            [
                nn.Dropout(self.dropout_prob)
                for _ in range(num_layers)
            ])

        # Dropout for MLP Module for all decoder layers
        self.ff_dropouts = nn.ModuleList(
            [
                nn.Dropout(self.dropout_prob)
                for _ in range(num_layers)
            ])

        # Per-layer (not shared) bbox heads, as in Deformable-DETR / RT-DETR
        self.bbox_heads = nn.ModuleList([
            nn.Sequential(
                nn.Linear(self.d_model, self.d_model),
                nn.ReLU(),
                nn.Linear(self.d_model, 4)
            )
            for _ in range(num_layers)
        ])

        # Shared Output norm for all decoder outputs
        self.output_norm = nn.LayerNorm(d_model)

        self._reset_parameters()

    def _reset_parameters(self):
        # Start every layer as an identity refinement: with a zeroed last layer
        # the predicted delta is 0, so layer 0 reproduces the encoder boxes and
        # the refinement chain has to earn any movement it makes.
        for bbox_head in self.bbox_heads:
            nn.init.constant_(bbox_head[-1].weight, 0.)
            nn.init.constant_(bbox_head[-1].bias, 0.)

    def forward(self, query_objects, query_pos_embed, reference_points,
                encoder_output, spatial_shapes, level_start_index):
        r"""
        :param query_objects: (B, num_queries, d_model) content queries
        :param query_pos_embed: (B, num_queries, d_model) positional queries
        :param reference_points: (B, num_queries, 4) cxcywh in [0, 1], the
            top-k boxes selected from the encoder
        :param encoder_output: (B, sum(H*W), d_model) flattened multi-scale memory
        :param spatial_shapes: (num_levels, 2) tensor of (H, W) per level
        :param level_start_index: (num_levels,) offsets into encoder_output
        :return: ``(outputs, reference_points)`` stacked over layers, of shapes
            (num_layers, B, num_queries, d_model) and (num_layers, B, num_queries, 4)
        """
        out = query_objects
        decoder_outputs = []
        updated_reference_points = []

        for i in range(self.num_layers):
            # Norm, Self Attention, Dropout and Residual
            in_attn = self.attn_norms[i](out)
            q = in_attn + query_pos_embed
            k = in_attn + query_pos_embed
            out_attn, _ = self.attns[i](
                query=q,
                key=k,
                value=in_attn
            )
            out_attn = self.attn_dropouts[i](out_attn)
            out = out + out_attn

            # Norm, Deformable Cross Attention, Dropout and Residual.
            # The cross-attention block owns its own output dropout.
            in_attn = self.cross_attn_norms[i](out)
            q = in_attn + query_pos_embed
            out_attn = self.cross_attns[i](
                query=q,
                reference_points=reference_points,
                input_flatten=encoder_output,
                input_spatial_shapes=spatial_shapes,
                input_level_start_index=level_start_index
            )
            out = out + out_attn

            # Norm, MLP, Dropout and Residual
            in_ff = self.ff_norms[i](out)
            out_ff = self.ffs[i](in_ff)
            out_ff = self.ff_dropouts[i](out_ff)
            out = out + out_ff
            layer_out = self.output_norm(out)
            decoder_outputs.append(layer_out)

            # ITERATIVE REFERENCE POINT REFINEMENT
            # Predict the offset (delta) from the current layer output
            delta_unsig = self.bbox_heads[i](layer_out)

            # Convert current reference points to inverse sigmoid space, add
            # delta, and sigmoid back to [0, 1] coordinate space.
            ref_unsig = inverse_sigmoid(reference_points)
            new_reference_points = (ref_unsig + delta_unsig).sigmoid()

            # Keep the *undetached* boxes for the loss...
            updated_reference_points.append(new_reference_points)
            # ...but detach the ones handed to the next layer, so gradients do
            # not flow back through the whole iterative update chain.
            reference_points = new_reference_points.detach()

        # Stack outputs for deep supervision loss computation
        output = torch.stack(decoder_outputs)
        ref_points_out = torch.stack(updated_reference_points)
        return output, ref_points_out

In [ ]:
class RepBlock(nn.Module):
    r"""
    RepVGG-style fusion block used inside CCFM: a 3x3 and a 1x1 branch (plus an
    identity branch when the channel count allows), summed and activated. At
    inference the three branches can be re-parameterised into a single 3x3 conv.

    :param in_channels: input channels
    :param out_channels: output channels
    """

    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, 3, 1, 1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.conv2 = nn.Conv2d(in_channels, out_channels, 1, 1, 0, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)

        self.skip_conn = (nn.BatchNorm2d(out_channels)
                          if in_channels == out_channels else None)

        self.relu = nn.ReLU()

    def forward(self, x):
        out1 = self.bn1(self.conv1(x))
        out2 = self.bn2(self.conv2(x))
        out3 = self.skip_conn(x) if self.skip_conn is not None else 0

        out = self.relu(out1 + out2 + out3)
        return out


class PositionEmbedding(nn.Module):
    r"""
    Sine/cosine embedding of a box, applied independently to each of its four
    normalised coordinates.

    :param num_pos_feats: features generated per coordinate
    """

    def __init__(self, num_pos_feats=128):
        super().__init__()
        self.num_pos_feats = num_pos_feats

    def forward(self, x):
        # x -> (B, num_queries, 4) with (cx, cy, w, h) normalised to [0, 1]
        scale = 2 * math.pi
        dim_t = torch.arange(self.num_pos_feats, dtype=torch.float32,
                             device=x.device)
        dim_t = 10000 ** (2 * (dim_t // 2) / self.num_pos_feats)

        x_sine = x.unsqueeze(-1) * scale / dim_t

        # Apply sin to even indices and cos to odd indices
        pos = torch.stack((x_sine[..., 0::2].sin(),
                           x_sine[..., 1::2].cos()), dim=-1).flatten(2)
        return pos  # -> (B, num_queries, 4 * num_pos_feats)


class RefPointsToQueryEmbed(nn.Module):
    r"""
    Turns reference boxes into positional queries, DAB-DETR style: sinusoidal
    encoding of (cx, cy, w, h) followed by a small MLP.

    :param embed_dim: output dimension (d_model)
    :param num_pos_feats: sine features per coordinate
    """

    def __init__(self, embed_dim=256, num_pos_feats=64):
        super().__init__()
        self.pos_emb = PositionEmbedding(num_pos_feats=num_pos_feats)
        # 4 coordinates x num_pos_feats each
        self.mlp = nn.Sequential(
            nn.Linear(4 * num_pos_feats, embed_dim),
            nn.ReLU(),
            nn.Linear(embed_dim, embed_dim)
        )

    def forward(self, reference_points):
        # 1. Generate sine/cosine positional features from reference points
        pos_embed = self.pos_emb(reference_points)
        # 2. Refine through an MLP
        return self.mlp(pos_embed)


class DETR(nn.Module):
    r"""
    RT-DETR. A forward pass goes through the following stages:
        1. Backbone (ResNet-34), taking S3 / S4 / S5
        2. 1x1 projection of each level to d_model
        3. AIFI - self attention on S5 only
        4. CCFM - top-down then bottom-up cross-scale fusion
        5. IoU-aware query selection - top-k encoder tokens become queries
        6. Deformable decoder with iterative box refinement
        7. Hungarian matching + set-prediction loss, per decoder layer plus the
           encoder selection head

    :param config: the ``model_params`` block of the config
    :param num_classes: including background
    :param bg_class_idx: index of the background class (0 or num_classes-1)
    """

    def __init__(self, config, num_classes, bg_class_idx):
        super().__init__()
        self.d_model = config['d_model']
        self.num_queries = config['num_queries']
        self.num_classes = num_classes
        self.num_decoder_layers = config['decoder_num_blocks']
        self.cls_cost_weight = config['cls_cost_weight']
        self.l1_cost_weight = config['l1_cost_weight']
        self.giou_cost_weight = config['giou_cost_weight']
        self.bg_cls_weight = config['bg_class_weight']
        self.nms_threshold = config['nms_threshold']
        self.bg_class_idx = bg_class_idx
        self.num_levels = 3
        valid_bg_idx = (self.bg_class_idx == 0 or
                        self.bg_class_idx == (self.num_classes - 1))
        assert valid_bg_idx, "Background can only be 0 or num_classes-1"

        # Keep the resnet module itself (not an nn.Sequential wrapper) so the
        # named stages stay reachable for the multi-scale forward below.
        backbone = resnet34(
            weights=torchvision.models.ResNet34_Weights.IMAGENET1K_V1,
            norm_layer=torchvision.ops.FrozenBatchNorm2d
        )
        del backbone.avgpool
        del backbone.fc
        self.backbone = backbone

        if config['freeze_backbone']:
            for param in self.backbone.parameters():
                param.requires_grad = False

        # ResNet-34 stage widths: layer2 -> 128, layer3 -> 256, layer4 -> 512
        self.backbone_proj = nn.ModuleList([
            nn.Conv2d(128, self.d_model, kernel_size=1),  # S3, stride 8
            nn.Conv2d(256, self.d_model, kernel_size=1),  # S4, stride 16
            nn.Conv2d(512, self.d_model, kernel_size=1),  # S5, stride 32
        ])

        # AIFI: intra-scale attention, on the highest level only
        self.s5_encoder = TransformerEncoder(
            num_layers=config['encoder_num_blocks'],
            num_heads=config['encoder_attn_heads'],
            d_model=config['d_model'],
            ff_inner_dim=config['ff_inner_dim'],
            dropout_prob=config['dropout_prob'])

        # ---- CCFM. These MUST be ModuleLists: as plain python lists their
        # parameters are invisible to .parameters(), .to(device) and state_dict.
        self.up_sampling = nn.ModuleList([
            nn.ConvTranspose2d(self.d_model, self.d_model, kernel_size=2,
                               stride=2, padding=0, bias=False),
            nn.ConvTranspose2d(self.d_model, self.d_model, kernel_size=2,
                               stride=2, padding=0, bias=False),
        ])

        self.down_sampling = nn.ModuleList([
            nn.Conv2d(self.d_model, self.d_model, kernel_size=3,
                      stride=2, padding=1, bias=False),
            nn.Conv2d(self.d_model, self.d_model, kernel_size=3,
                      stride=2, padding=1, bias=False),
        ])

        self.rep_blocks_up = nn.ModuleList([
            RepBlock(in_channels=2 * self.d_model, out_channels=self.d_model),
            RepBlock(in_channels=2 * self.d_model, out_channels=self.d_model),
        ])

        self.rep_blocks_down = nn.ModuleList([
            RepBlock(in_channels=2 * self.d_model, out_channels=self.d_model),
            RepBlock(in_channels=2 * self.d_model, out_channels=self.d_model),
        ])

        # ---- Query selection heads, run over every encoder token.
        self.enc_class_head = nn.Linear(self.d_model, self.num_classes)
        self.enc_bbox_head = nn.Sequential(
            nn.Linear(self.d_model, self.d_model),
            nn.ReLU(),
            nn.Linear(self.d_model, 4),
        )
        # Zeroed last layer => the encoder starts by predicting the anchors
        # themselves, and has to learn any deviation from them.
        nn.init.constant_(self.enc_bbox_head[-1].weight, 0.)
        nn.init.constant_(self.enc_bbox_head[-1].bias, 0.)

        # Positional queries derived from the selected reference boxes
        self.ref_to_query_embed = RefPointsToQueryEmbed(embed_dim=self.d_model)

        self.decoder = DeformableTransformerDecoder(
            num_layers=config['decoder_num_blocks'],
            num_heads=config['decoder_attn_heads'],
            d_model=config['d_model'],
            ff_inner_dim=config['ff_inner_dim'],
            dropout_prob=config['dropout_prob'],
            num_levels=self.num_levels,
            num_points=config.get('decoder_num_points', 4)
        )

        # Decoder classification head (the box heads live inside the decoder,
        # one per layer, because refinement is iterative).
        self.class_mlp = nn.Linear(self.d_model, self.num_classes)

        # Down-weight background so the loss is not dominated by it. A buffer so
        # it follows the model onto the right device, and is built once rather
        # than once per decoder layer per forward.
        cls_weights = torch.ones(self.num_classes)
        cls_weights[self.bg_class_idx] = self.bg_cls_weight
        self.register_buffer('cls_weights', cls_weights)

        # Anchors depend only on the feature-map shapes, so cache them.
        self._anchor_cache = {}

    def backbone_forward(self, x):
        r"""
        :param x: (B, 3, H, W)
        :return: S3 (stride 8, 128ch), S4 (stride 16, 256ch), S5 (stride 32, 512ch)
        """
        backbone = self.backbone
        x = backbone.conv1(x)
        x = backbone.bn1(x)
        x = backbone.relu(x)
        x = backbone.maxpool(x)

        x = backbone.layer1(x)
        s3 = backbone.layer2(x)   # stride 8,  128 channels
        s4 = backbone.layer3(s3)  # stride 16, 256 channels
        s5 = backbone.layer4(s4)  # stride 32, 512 channels

        return s3, s4, s5

    def get_anchors(self, spatial_shapes, device, dtype):
        r"""Cached :func:`generate_anchors` keyed on shape + device."""
        key = (tuple(spatial_shapes.flatten().tolist()), str(device), dtype)
        if key not in self._anchor_cache:
            self._anchor_cache[key] = generate_anchors(
                spatial_shapes, dtype=dtype, device=device)
        return self._anchor_cache[key]

    def set_loss(self, cls_output, bbox_output, targets):
        r"""
        Hungarian matching plus the DETR set-prediction loss, for one set of
        predictions. Called once per decoder layer and once for the encoder
        query-selection head.

        :param cls_output: (B, num_queries, num_classes) logits
        :param bbox_output: (B, num_queries, 4) cxcywh in [0, 1]
        :param targets: list of B dicts with 'boxes' (N_i, 4) xyxy normalised
            and 'labels' (N_i,)
        :return: ``(loss_cls, loss_bbox, loss_giou)`` scalars
        """
        batch_size, num_queries, num_classes = cls_output.shape

        with torch.no_grad():
            # Concat all prediction boxes and class prob together
            class_prob = cls_output.reshape((-1, num_classes))
            class_prob = class_prob.softmax(dim=-1)
            # class_prob -> (B*num_queries, num_classes)

            pred_boxes = bbox_output.reshape((-1, 4))
            # pred_boxes -> (B*num_queries, 4)

            # Concat all target boxes and labels also together
            target_labels = torch.cat([target["labels"] for target in targets])
            target_boxes = torch.cat([target["boxes"] for target in targets])
            # len(target_labels) -> num_targets_for_entire_batch
            # target_boxes -> (num_targets_for_entire_batch, 4)

            # Classification Cost
            cost_classification = -class_prob[:, target_labels]
            # cost_cls -> (B*num_queries, num_targets_for_entire_batch)

            # We predict cx,cy,w,h , we need to convert to x1y1x2y2 for giou.
            # Don't need to convert targets as they are already in x1y1x2y2
            pred_boxes_x1y1x2y2 = torchvision.ops.box_convert(
                pred_boxes, 'cxcywh', 'xyxy')

            cost_localization_l1 = torch.cdist(
                pred_boxes_x1y1x2y2,
                target_boxes,
                p=1
            )
            # cost_l1 -> (B*num_queries, num_targets_for_entire_batch)

            cost_localization_giou = -torchvision.ops.generalized_box_iou(
                pred_boxes_x1y1x2y2,
                target_boxes
            )
            # cost_giou -> (B*num_queries, num_targets_for_entire_batch)

            total_cost = (self.l1_cost_weight * cost_localization_l1
                          + self.cls_cost_weight * cost_classification
                          + self.giou_cost_weight * cost_localization_giou)

            # linear_sum_assignment is scipy, so the cost has to come back to cpu
            total_cost = total_cost.reshape(batch_size, num_queries, -1).cpu()
            # total_cost -> (B, num_queries, num_targets_for_entire_batch)

            num_targets_per_image = [len(target["labels"]) for target in targets]
            total_cost_per_batch_image = total_cost.split(
                num_targets_per_image,
                dim=-1
            )
            # total_cost_per_batch_image[i] = (B,num_queries,num_targets_ith_image)

            match_indices = []
            for batch_idx in range(batch_size):
                batch_idx_assignments = linear_sum_assignment(
                    total_cost_per_batch_image[batch_idx][batch_idx]
                )
                batch_idx_pred, batch_idx_target = batch_idx_assignments
                # len(batch_idx_pred) = num_targets_ith_image
                match_indices.append((torch.as_tensor(batch_idx_pred,
                                                      dtype=torch.int64),
                                      torch.as_tensor(batch_idx_target,
                                                      dtype=torch.int64)))
                # match_indices -> [
                #   ([pred_box_a1, ...],[target_box_i1, ...]),
                #   ([pred_box_a2, ...],[target_box_i2, ...]),
                #   ... assignment pairs for ith batch image
                #   ]

        # pred_batch_idxs are batch indexes for each assignment pair
        pred_batch_idxs = torch.cat([
            torch.ones_like(pred_idx) * i
            for i, (pred_idx, _) in enumerate(match_indices)
        ])
        # pred_batch_idxs -> (num_targets_for_entire_batch, )

        # pred_query_idx are prediction box indexes(out of num_queries)
        # for each assignment pair
        pred_query_idx = torch.cat([pred_idx for (pred_idx, _) in match_indices])
        # pred_query_idx -> (num_targets_for_entire_batch, )

        # For all assigned prediction boxes, get the target label
        valid_obj_target_cls = torch.cat([
            target["labels"][target_obj_idx]
            for target, (_, target_obj_idx) in zip(targets, match_indices)
        ])
        # valid_obj_target_cls -> (num_targets_for_entire_batch, )

        # Initialize target class for all predicted boxes to be background class
        target_classes = torch.full(
            cls_output.shape[:2],
            fill_value=self.bg_class_idx,
            dtype=torch.int64,
            device=cls_output.device
        )
        # target_classes -> (B, num_queries)

        # For predicted boxes that were assigned to some target,
        # update their target label accordingly
        target_classes[(pred_batch_idxs, pred_query_idx)] = (
            valid_obj_target_cls.to(target_classes.device))

        # Compute classification loss
        loss_cls = torch.nn.functional.cross_entropy(
            cls_output.reshape(-1, self.num_classes),
            target_classes.reshape(-1),
            self.cls_weights)

        # Get pred box coordinates for all matched pred boxes
        matched_pred_boxes = bbox_output[pred_batch_idxs, pred_query_idx]
        # matched_pred_boxes -> (num_targets_for_entire_batch, 4)

        # Get target box coordinates for all matched target boxes
        matched_target_boxes = torch.cat([
            target['boxes'][target_obj_idx]
            for target, (_, target_obj_idx) in zip(targets, match_indices)],
            dim=0
        )
        # matched_target_boxes -> (num_targets_for_entire_batch, 4)

        # Convert matched pred boxes to x1y1x2y2 format
        matched_pred_boxes_x1y1x2y2 = torchvision.ops.box_convert(
            matched_pred_boxes, 'cxcywh', 'xyxy')

        # An image batch can legitimately contain no boxes at all; clamp the
        # normalizer so the loss is 0 rather than NaN.
        num_boxes = max(matched_pred_boxes.shape[0], 1)

        # Don't need to convert target boxes as they are in x1y1x2y2 format.
        # Compute L1 Localization loss
        loss_bbox = torch.nn.functional.l1_loss(
            matched_pred_boxes_x1y1x2y2,
            matched_target_boxes,
            reduction='none')
        loss_bbox = loss_bbox.sum() / num_boxes

        # Compute GIoU loss
        loss_giou = torchvision.ops.generalized_box_iou_loss(
            matched_pred_boxes_x1y1x2y2,
            matched_target_boxes
        )
        loss_giou = loss_giou.sum() / num_boxes

        return loss_cls, loss_bbox, loss_giou

    def forward(self, x, targets=None, score_thresh=0, use_nms=False):
        # x -> (B, C, H, W)
        # default d_model - 256
        # default C - 3
        # default H,W - 640,640
        # S3,S4,S5 feature maps -> 80x80, 40x40, 20x20

        s3, s4, s5 = self.backbone_forward(x)

        s3, s4, s5 = [
            proj(feat)
            for proj, feat in zip(self.backbone_proj, [s3, s4, s5])
        ]
        # each -> (B, d_model, feat_h, feat_w)

        # ---------------- AIFI (S5 only) ----------------
        batch_size, d_model, feat_h, feat_w = s5.shape
        spatial_pos_embed = get_spatial_position_embedding(self.d_model, s5)
        # spatial_pos_embed -> (feat_h * feat_w, d_model)

        s5_flat = s5.reshape(batch_size, d_model,
                             feat_h * feat_w).transpose(1, 2)
        # s5_flat -> (B, feat_h*feat_w, d_model)

        enc_output, enc_attn_weights = self.s5_encoder(s5_flat,
                                                       spatial_pos_embed)
        # enc_output -> (B, feat_h*feat_w, d_model)
        # enc_attn_weights -> (num_encoder_layers, B, feat_h*feat_w, feat_h*feat_w)

        # Back to a 2D map so the convolutional neck can consume it
        f5 = enc_output.transpose(1, 2).reshape(batch_size, d_model,
                                                feat_h, feat_w)
        # ---------------- AIFI end ----------------

        # ---------------- CCFM ----------------
        # Top-down: F5 -> P4 -> P3
        up2 = self.up_sampling[0](f5)
        up2 = torch.cat((up2, s4), dim=1)
        up2 = self.rep_blocks_up[0](up2)

        up3 = self.up_sampling[1](up2)
        up3 = torch.cat((up3, s3), dim=1)
        up3 = self.rep_blocks_up[1](up3)

        # Bottom-up: P3 -> N4 -> N5
        down2 = self.down_sampling[0](up3)
        down2 = torch.cat((down2, up2), dim=1)
        down2 = self.rep_blocks_down[0](down2)

        down1 = self.down_sampling[1](down2)
        down1 = torch.cat((down1, f5), dim=1)
        down1 = self.rep_blocks_down[1](down1)
        # ---------------- CCFM end ----------------

        # Flatten the three levels into one token sequence, finest level first
        levels = [up3, down2, down1]  # strides 8, 16, 32
        encoder_output = torch.cat(
            [level.flatten(2).transpose(1, 2) for level in levels], dim=1)
        # encoder_output -> (B, sum(H*W), d_model), e.g. (B, 8400, 256)

        spatial_shapes = torch.as_tensor(
            [[level.shape[2], level.shape[3]] for level in levels],
            dtype=torch.long, device=encoder_output.device)
        # spatial_shapes -> (num_levels, 2), e.g. [[80,80],[40,40],[20,20]]
        level_start_index = torch.cat([
            spatial_shapes.new_zeros(1),
            spatial_shapes.prod(dim=1).cumsum(0)[:-1]
        ])
        # level_start_index -> (num_levels,), e.g. [0, 6400, 8000]

        # ---------------- IoU-aware query selection ----------------
        anchors, valid_mask = self.get_anchors(spatial_shapes,
                                               encoder_output.device,
                                               encoder_output.dtype)
        # Border cells whose anchor falls outside the image are zeroed out
        memory = encoder_output * valid_mask

        enc_logits = self.enc_class_head(memory)
        # enc_logits -> (B, sum(H*W), num_classes)
        enc_boxes = (self.enc_bbox_head(memory) + anchors).sigmoid()
        # enc_boxes -> (B, sum(H*W), 4) cxcywh in [0, 1]

        # Score by the best *foreground* class. Including background here would
        # rank tokens by how confidently empty they are.
        if self.bg_class_idx == 0:
            scores = enc_logits[..., 1:].max(dim=-1).values
        else:
            scores = enc_logits[..., :-1].max(dim=-1).values
        # scores -> (B, sum(H*W))

        topk_indices = torch.topk(scores, k=self.num_queries, dim=1).indices
        # topk_indices -> (B, num_queries)

        gather_d = topk_indices.unsqueeze(-1).expand(-1, -1, self.d_model)
        gather_c = topk_indices.unsqueeze(-1).expand(-1, -1, self.num_classes)
        gather_4 = topk_indices.unsqueeze(-1).expand(-1, -1, 4)

        # Content queries are the encoder features of the selected tokens,
        # detached so the decoder does not backprop into the memory through them
        query_objects = torch.gather(encoder_output, 1, gather_d).detach()

        # These two stay attached: they carry the gradient for the encoder
        # auxiliary loss, which is the only thing that trains the selection head
        enc_topk_logits = torch.gather(enc_logits, 1, gather_c)
        enc_topk_boxes = torch.gather(enc_boxes, 1, gather_4)

        # The decoder refines a detached copy
        reference_points = enc_topk_boxes.detach()
        query_pos_embed = self.ref_to_query_embed(reference_points)
        # query_pos_embed -> (B, num_queries, d_model)

        decoder_outputs, updated_reference_points = self.decoder(
            query_objects,
            query_pos_embed,
            reference_points,
            encoder_output,
            spatial_shapes,
            level_start_index,
        )
        # decoder_outputs -> (num_decoder_layers, B, num_queries, d_model)
        # updated_reference_points -> (num_decoder_layers, B, num_queries, 4)

        cls_output = self.class_mlp(decoder_outputs)
        # cls_output -> (num_decoder_layers, B, num_queries, num_classes)
        bbox_output = updated_reference_points
        # bbox_output -> (num_decoder_layers, B, num_queries, 4), cxcywh in [0,1]

        losses = defaultdict(list)
        detections = []
        detr_output = {}

        if targets is not None:
            # Deep supervision: match and score every decoder layer...
            for decoder_idx in range(self.num_decoder_layers):
                loss_cls, loss_bbox, loss_giou = self.set_loss(
                    cls_output[decoder_idx], bbox_output[decoder_idx], targets)
                losses['classification'].append(loss_cls * self.cls_cost_weight)
                losses['bbox_regression'].append(
                    loss_bbox * self.l1_cost_weight
                    + loss_giou * self.giou_cost_weight
                )

            # ...plus one extra term on the encoder query-selection head.
            # Without this, enc_class_head feeds only a non-differentiable
            # topk and receives no gradient at all, so query selection would
            # stay frozen at its random initialisation forever.
            # NOTE: appended to the same two lists the training loop already
            # sums, so the last entry of each is the encoder head rather than a
            # decoder layer.
            loss_cls, loss_bbox, loss_giou = self.set_loss(
                enc_topk_logits, enc_topk_boxes, targets)
            losses['classification'].append(loss_cls * self.cls_cost_weight)
            losses['bbox_regression'].append(
                loss_bbox * self.l1_cost_weight
                + loss_giou * self.giou_cost_weight
            )

            detr_output['loss'] = losses

        if not self.training:
            # For inference we are only interested in last layer outputs
            final_cls_output = cls_output[-1]
            final_bbox_output = bbox_output[-1]
            # final_cls_output -> (B, num_queries, num_classes)
            # final_bbox_output -> (B, num_queries, 4)

            prob = torch.nn.functional.softmax(final_cls_output, -1)

            # Get all query boxes and their best fg class as label
            if self.bg_class_idx == 0:
                scores, labels = prob[..., 1:].max(-1)
                labels = labels + 1
            else:
                scores, labels = prob[..., :-1].max(-1)

            # convert to x1y1x2y2 format
            boxes = torchvision.ops.box_convert(final_bbox_output,
                                                'cxcywh',
                                                'xyxy')

            for batch_idx in range(boxes.shape[0]):
                scores_idx = scores[batch_idx]
                labels_idx = labels[batch_idx]
                boxes_idx = boxes[batch_idx]

                # Low score filtering
                keep_idxs = scores_idx >= score_thresh
                scores_idx = scores_idx[keep_idxs]
                boxes_idx = boxes_idx[keep_idxs]
                labels_idx = labels_idx[keep_idxs]

                # NMS filtering
                if use_nms:
                    keep_idxs = torchvision.ops.batched_nms(
                        boxes_idx,
                        scores_idx,
                        labels_idx,
                        iou_threshold=self.nms_threshold)
                    scores_idx = scores_idx[keep_idxs]
                    boxes_idx = boxes_idx[keep_idxs]
                    labels_idx = labels_idx[keep_idxs]
                detections.append(
                    {
                        "boxes": boxes_idx,
                        "scores": scores_idx,
                        "labels": labels_idx,
                    }
                )

            detr_output['detections'] = detections
            detr_output['enc_attn'] = enc_attn_weights
        return detr_output

## Smoke test

In [ ]:
# ---------------------------------------------------------------------------
# Smoke test: shapes, finite losses, full gradient coverage, and the eval path.
# CPU-only, no dataset needed.
# ---------------------------------------------------------------------------
print('MSDeformAttn backend in use:', _MSDA_BACKEND)

with open('rt_detr_config.yaml', 'r') as f:
    smoke_config = yaml.safe_load(f)

dataset_config = smoke_config['dataset_params']
model_config = smoke_config['model_params']

torch.manual_seed(0)
model = DETR(model_config,
             num_classes=dataset_config['num_classes'],
             bg_class_idx=dataset_config['bg_class_idx'])
model.train()

im_size = dataset_config['im_size']
batch_size = 2
num_classes = dataset_config['num_classes']
num_queries = model_config['num_queries']
d_model = model_config['d_model']
num_decoder_layers = model_config['decoder_num_blocks']

x = torch.randn(batch_size, 3, im_size, im_size)

# Ragged target counts on purpose: the second image has NO boxes at all, which
# is what exercises the zero-target normalizer.
targets = [
    {
        'boxes': torch.tensor([[0.10, 0.10, 0.40, 0.50],
                               [0.30, 0.20, 0.90, 0.85],
                               [0.55, 0.60, 0.70, 0.95]]),
        'labels': torch.tensor([3, 11, 7]),
    },
    {
        'boxes': torch.zeros((0, 4)),
        'labels': torch.zeros((0,), dtype=torch.int64),
    },
]

# ---- 1. Backbone shapes -----------------------------------------------------
s3, s4, s5 = model.backbone_forward(x)
assert s3.shape == (batch_size, 128, im_size // 8, im_size // 8), s3.shape
assert s4.shape == (batch_size, 256, im_size // 16, im_size // 16), s4.shape
assert s5.shape == (batch_size, 512, im_size // 32, im_size // 32), s5.shape
print('backbone  S3/S4/S5 :', tuple(s3.shape), tuple(s4.shape), tuple(s5.shape))

expected_tokens = sum((im_size // stride) ** 2 for stride in (8, 16, 32))
print('expected encoder tokens :', expected_tokens)

# ---- 2. Parameter registration ---------------------------------------------
# A plain python list of submodules would leave the whole CCFM neck out of
# .parameters(); check the neck is actually registered.
param_names = {name for name, _ in model.named_parameters()}
for expected in ['up_sampling.0.weight', 'down_sampling.1.weight',
                 'rep_blocks_up.0.conv1.weight',
                 'rep_blocks_down.1.conv2.weight']:
    assert expected in param_names, 'CCFM param missing: {}'.format(expected)
print('registered params :', len(param_names),
      '| trainable tensors :',
      sum(1 for p in model.parameters() if p.requires_grad))

# ---- 3. Training forward ----------------------------------------------------
out = model(x, targets)
losses = out['loss']

# num_decoder_layers deep-supervision terms + 1 encoder query-selection term
assert len(losses['classification']) == num_decoder_layers + 1, \
    len(losses['classification'])
assert len(losses['bbox_regression']) == num_decoder_layers + 1, \
    len(losses['bbox_regression'])

for key, terms in losses.items():
    for i, term in enumerate(terms):
        tag = 'encoder' if i == num_decoder_layers else 'layer {}'.format(i)
        assert torch.isfinite(term), '{} {} is not finite: {}'.format(
            key, tag, term)
    print('{:16s}:'.format(key),
          ' '.join('{:.3f}'.format(t.item()) for t in terms),
          '  (last term = encoder head)')

total_loss = sum(losses['classification']) + sum(losses['bbox_regression'])
assert torch.isfinite(total_loss)
print('total loss :', float(total_loss.detach()))
# At init every bbox head is zero-initialised, so all layers reproduce the
# encoder boxes exactly and every bbox term is identical. Section 8 checks they
# separate once gradients start flowing - that is the real refinement test.

# ---- 4. Backward + gradient coverage ---------------------------------------
total_loss.backward()

no_grad = [name for name, p in model.named_parameters()
           if p.requires_grad and p.grad is None]
assert not no_grad, 'params with no gradient at all:\n  ' + '\n  '.join(no_grad)

# enc_class_head in particular must be non-zero -- that is the whole point of
# the encoder auxiliary loss.
assert model.enc_class_head.weight.grad.abs().sum() > 0, \
    'encoder query-selection head received no gradient'
print('all', sum(1 for p in model.parameters() if p.requires_grad),
      'trainable tensors received a gradient '
      '(incl. enc_class_head, CCFM neck, deformable offsets)')

model.zero_grad(set_to_none=True)
model.eval()

# ---- 5. Neck internals, re-derived for the shape asserts -------------------
with torch.no_grad():
    s3p, s4p, s5p = [proj(feat) for proj, feat in
                     zip(model.backbone_proj, model.backbone_forward(x))]
    pos = get_spatial_position_embedding(d_model, s5p)
    b, c, h, w = s5p.shape
    enc, _ = model.s5_encoder(s5p.reshape(b, c, h * w).transpose(1, 2), pos)
    f5 = enc.transpose(1, 2).reshape(b, c, h, w)

    up2 = model.rep_blocks_up[0](torch.cat((model.up_sampling[0](f5), s4p), 1))
    up3 = model.rep_blocks_up[1](torch.cat((model.up_sampling[1](up2), s3p), 1))
    dn2 = model.rep_blocks_down[0](
        torch.cat((model.down_sampling[0](up3), up2), 1))
    dn1 = model.rep_blocks_down[1](
        torch.cat((model.down_sampling[1](dn2), f5), 1))

    levels = [up3, dn2, dn1]
    memory = torch.cat([lv.flatten(2).transpose(1, 2) for lv in levels], dim=1)
    shapes = torch.as_tensor([[lv.shape[2], lv.shape[3]] for lv in levels],
                             dtype=torch.long)
    starts = torch.cat([shapes.new_zeros(1), shapes.prod(1).cumsum(0)[:-1]])

assert memory.shape == (batch_size, expected_tokens, d_model), memory.shape
assert shapes.tolist() == [[80, 80], [40, 40], [20, 20]], shapes.tolist()
assert starts.tolist() == [0, 6400, 8000], starts.tolist()
print('encoder_output    :', tuple(memory.shape))
print('spatial_shapes    :', shapes.tolist())
print('level_start_index :', starts.tolist())

anchors, valid_mask = model.get_anchors(shapes, memory.device, memory.dtype)
assert anchors.shape == (1, expected_tokens, 4), anchors.shape
print('anchors           :', tuple(anchors.shape),
      '| valid:', int(valid_mask.sum()), 'of', expected_tokens)

# ---- 6. Query selection + decoder, driven directly -------------------------
with torch.no_grad():
    masked_memory = memory * valid_mask
    enc_logits = model.enc_class_head(masked_memory)
    enc_boxes = (model.enc_bbox_head(masked_memory) + anchors).sigmoid()

    fg_scores = enc_logits[..., 1:].max(dim=-1).values
    topk_indices = torch.topk(fg_scores, k=num_queries, dim=1).indices
    gather_d = topk_indices.unsqueeze(-1).expand(-1, -1, d_model)
    gather_4 = topk_indices.unsqueeze(-1).expand(-1, -1, 4)

    query_objects = torch.gather(memory, 1, gather_d)
    reference_points = torch.gather(enc_boxes, 1, gather_4)
    query_pos_embed = model.ref_to_query_embed(reference_points)

    dec_out, ref_out = model.decoder(query_objects, query_pos_embed,
                                     reference_points, memory, shapes, starts)

assert enc_boxes.shape == (batch_size, expected_tokens, 4), enc_boxes.shape
assert torch.all((enc_boxes >= 0) & (enc_boxes <= 1)), \
    'encoder boxes escaped [0, 1]'
assert query_pos_embed.shape == (batch_size, num_queries, d_model), \
    query_pos_embed.shape
assert dec_out.shape == (num_decoder_layers, batch_size, num_queries, d_model), \
    dec_out.shape
assert ref_out.shape == (num_decoder_layers, batch_size, num_queries, 4), \
    ref_out.shape
assert torch.all((ref_out >= 0) & (ref_out <= 1)), \
    'reference points escaped [0, 1]'
assert torch.isfinite(ref_out).all(), 'reference points went non-finite'
print('enc_boxes         :', tuple(enc_boxes.shape), '| all in [0, 1]')
print('query_pos_embed   :', tuple(query_pos_embed.shape))
print('decoder_outputs   :', tuple(dec_out.shape))
print('reference_points  :', tuple(ref_out.shape),
      '| range [{:.4f}, {:.4f}]'.format(float(ref_out.min()),
                                        float(ref_out.max())))

# ---- 7. Eval / detection path ----------------------------------------------
with torch.no_grad():
    eval_out = model(x, score_thresh=0.0, use_nms=False)

detections = eval_out['detections']
assert len(detections) == batch_size
for det in detections:
    assert det['boxes'].shape == (num_queries, 4), det['boxes'].shape
    assert det['scores'].shape == (num_queries,), det['scores'].shape
    assert det['labels'].shape == (num_queries,), det['labels'].shape
    assert torch.all(det['labels'] >= 1) and torch.all(det['labels'] < num_classes)
print('detections        :', tuple(detections[0]['boxes'].shape),
      '| labels in [{}, {}]'.format(int(detections[0]['labels'].min()),
                                    int(detections[0]['labels'].max())))

# NMS branch, since it takes a different code path. Threshold stays at 0 -- at
# random init the best foreground softmax score is around 1/num_classes, so any
# real threshold would filter everything out and never reach batched_nms.
with torch.no_grad():
    nms_out = model(x, score_thresh=0.0, use_nms=True)
kept = [int(d['boxes'].shape[0]) for d in nms_out['detections']]
assert all(0 < k <= num_queries for k in kept), kept
print('after nms         :', kept, 'of', num_queries, 'boxes kept')

# Eval forward WITH targets (the validation-loss path) must also work
with torch.no_grad():
    val_out = model(x, targets)
assert 'loss' in val_out and 'detections' in val_out
assert all(torch.isfinite(t) for t in val_out['loss']['classification'])
print('eval-with-targets : loss + detections both produced')

# ---- 8. Does it actually learn? --------------------------------------------
# Overfit this one batch for a few steps. Two things matter here:
#   * the total loss goes down at all, and
#   * the per-layer bbox terms stop being identical and start *decreasing with
#     depth* -- which is the observable signature of iterative refinement
#     actually being wired up rather than each layer re-predicting the same box.
model.train()
optimizer = torch.optim.AdamW(
    [p for p in model.parameters() if p.requires_grad], lr=1e-4)

first_total, last_terms = None, None
print('\nstep |     total |   cls sum |  box sum')
for step in range(6):
    step_losses = model(x, targets)['loss']
    step_total = (sum(step_losses['classification'])
                  + sum(step_losses['bbox_regression']))
    optimizer.zero_grad()
    step_total.backward()
    torch.nn.utils.clip_grad_norm_(
        [p for p in model.parameters() if p.requires_grad], 0.1)
    optimizer.step()

    if first_total is None:
        first_total = float(step_total.detach())
    last_terms = [float(t.detach()) for t in step_losses['bbox_regression']]
    print('{:4d} | {:9.4f} | {:9.4f} | {:8.4f}'.format(
        step, float(step_total.detach()),
        float(sum(step_losses['classification']).detach()),
        float(sum(step_losses['bbox_regression']).detach())))

last_total = float(step_total.detach())
assert last_total < first_total, \
    'loss did not decrease: {:.4f} -> {:.4f}'.format(first_total, last_total)

decoder_terms, encoder_term = last_terms[:-1], last_terms[-1]
print('per-layer bbox_regression :',
      ' '.join('{:.3f}'.format(t) for t in decoder_terms),
      '| encoder {:.3f}'.format(encoder_term))
assert len(set('{:.4f}'.format(t) for t in decoder_terms)) > 1, \
    'all decoder layers give an identical box loss -- refinement is not wired up'
print('loss {:.2f} -> {:.2f}; decoder layers differentiated'.format(
    first_total, last_total))

print('\nSMOKE TEST PASSED')

## Train on VOC

Everything below is driven by `rt_detr_config.yaml` - epochs, batch size,
learning-rate schedule, worker count, validation split and checkpoint naming.

Checkpoints use the same format as `detr_mor/engine/checkpoint.py`, so
`{task_name}/best_{ckpt_name}` stays loadable by `scripts/evaluate.py`. The run
resumes automatically from `{task_name}/{ckpt_name}` if it is interrupted.

In [ ]:
# ---------------------------------------------------------------------------
# Training on VOC, driven entirely by rt_detr_config.yaml.
#
# Checkpoint format matches detr_mor/engine/checkpoint.py, so the files written
# here stay loadable by scripts/evaluate.py and by the rest of the repo.
#
# Heads-up on memory: the config asks for batch_size 80 at 640x640. The CCFM
# carries a (B, 256, 80, 80) activation and AIFI a (B, 8, 400, 400) attention
# matrix, so 80 will not fit on a single consumer GPU. If you OOM, drop
# batch_size and raise acc_steps by the same factor - the effective batch, and
# therefore the optimizer trajectory, is batch_size * acc_steps.
# ---------------------------------------------------------------------------
import platform
import time


def collate_function(data):
    r"""
    Detection targets are ragged (a different number of objects per image), so we
    cannot stack them. Transpose the batch into tuples instead:
    ``(images, targets, filenames)``.
    """
    return tuple(zip(*data))


def set_seed(seed):
    r"""Seed torch, numpy and python's rng."""
    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)


def resolve_num_workers(train_config):
    r"""
    Worker count for the loaders.

    Augmentation (zoom-out, IoU crop, photometric distort, 640px resize) runs on
    the CPU per image, so with ``num_workers=0`` it blocks the training step and
    the GPU starves. Workers only pay off when something else can use the time,
    hence the CUDA-dependent default.

    Forced to 0 in a Windows notebook: DataLoader workers use *spawn* there, and
    the child process cannot unpickle ``VOCDataset`` because the class lives in
    the notebook's ``__main__`` rather than an importable module. The symptom is
    a hang or ``Can't get attribute 'VOCDataset'``. On Linux / Kaggle the fork
    start method has no such problem and the config value is used as written.

    :param train_config: config['train_params']
    """
    default_workers = 4 if torch.cuda.is_available() else 0
    num_workers = train_config.get('num_workers', default_workers)
    if num_workers > 0 and platform.system() == 'Windows':
        print('Windows notebook: forcing num_workers {} -> 0 (spawn cannot '
              'pickle VOCDataset out of __main__). The config value is honoured '
              'on Linux / Kaggle.'.format(num_workers))
        num_workers = 0
    return num_workers


def build_train_val_loaders(dataset_config, train_config, generator=None):
    r"""
    Build the train/val DataLoaders by randomly splitting the train imageset.

    :param dataset_config: config['dataset_params']
    :param train_config: config['train_params']
    :param generator: optional torch.Generator, for a reproducible split
    :return: (train_loader, val_loader, dataset)
    """
    voc = VOCDataset('train',
                     im_sets=dataset_config['train_im_sets'],
                     im_size=dataset_config['im_size'])

    val_fraction = train_config.get('val_split', 0.2)
    val_size = int(val_fraction * len(voc))
    train_size = len(voc) - val_size

    split_kwargs = {} if generator is None else {'generator': generator}
    train_subset, val_subset = random_split(voc, [train_size, val_size],
                                            **split_kwargs)

    num_workers = resolve_num_workers(train_config)
    loader_kwargs = {'num_workers': num_workers,
                     'pin_memory': torch.cuda.is_available()}
    if num_workers > 0:
        # Respawning workers every epoch costs seconds per epoch at this size.
        loader_kwargs['persistent_workers'] = True

    train_loader = DataLoader(train_subset,
                              batch_size=train_config['batch_size'],
                              shuffle=True,
                              collate_fn=collate_function,
                              **loader_kwargs)
    val_loader = DataLoader(val_subset,
                            batch_size=train_config['batch_size'],
                            shuffle=False,
                            collate_fn=collate_function,
                            **loader_kwargs)

    print('Dataset: {} images -> {} train / {} val | batch_size {} | '
          'num_workers {}'.format(len(voc), train_size, val_size,
                                  train_config['batch_size'], num_workers))
    return train_loader, val_loader, voc


def targets_to_device(targets, device):
    r"""Move 'boxes' and 'labels' onto ``device``, in the dtypes the loss wants."""
    for target in targets:
        target['boxes'] = target['boxes'].float().to(device)
        target['labels'] = target['labels'].long().to(device)
    return targets


def batch_images_to_device(images, device):
    r"""Stack the per-image tensors from ``collate_function`` into (B, C, H, W)."""
    return torch.stack([im.float().to(device) for im in images], dim=0)


def save_checkpoint(model, optimizer, scheduler, epoch, loss, steps, path,
                    extra=None):
    r"""
    Write a resumable checkpoint, in the same format as
    ``detr_mor.engine.checkpoint.save_checkpoint``.

    :param epoch: index of the epoch that just finished
    :param steps: global step counter
    :param extra: optional dict of extra fields (e.g. the losses that made this
        the best epoch)
    """
    os.makedirs(os.path.dirname(path) or '.', exist_ok=True)
    checkpoint = {
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict(),
        'loss': loss.item() if torch.is_tensor(loss) else loss,
        'steps': steps,
    }
    if extra:
        checkpoint.update(extra)
    torch.save(checkpoint, path)
    print('Checkpoint saved at epoch {}'.format(epoch))


def load_checkpoint(path, model, optimizer=None, scheduler=None,
                    map_location=None):
    r"""
    Restore a checkpoint. Returns ``(start_epoch, steps)``.
    """
    checkpoint = torch.load(path, map_location=map_location, weights_only=False)

    if isinstance(checkpoint, dict) and 'model_state_dict' in checkpoint:
        model.load_state_dict(checkpoint['model_state_dict'])
        if optimizer is not None and 'optimizer_state_dict' in checkpoint:
            optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
        if scheduler is not None and 'scheduler_state_dict' in checkpoint:
            scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
        return checkpoint.get('epoch', 0), checkpoint.get('steps', 0)

    model.load_state_dict(checkpoint)
    print('Loaded a bare state_dict; optimizer/scheduler state and the epoch '
          'counter are unavailable, so training restarts from epoch 0.')
    return 0, 0


def read_min_val_loss(path, map_location='cpu'):
    r"""Recover ``min_val_loss`` from an existing best checkpoint, else ``inf``."""
    if not os.path.exists(path):
        return float('inf')
    checkpoint = torch.load(path, map_location=map_location, weights_only=False)
    if not isinstance(checkpoint, dict):
        return float('inf')
    return checkpoint.get('min_val_loss', float('inf'))


def train_one_epoch(model, loader, optimizer, scheduler, device, train_config,
                    steps=0):
    r"""
    Run one training epoch.

    Gradients accumulate over ``acc_steps`` batches before each optimizer step;
    the last batch of the epoch always steps, so a partial accumulation window
    is never dropped.

    :param steps: global step counter carried across epochs
    :return: (mean classification loss, mean localization loss, steps, last loss)
    """
    model.train()
    acc_steps = train_config['acc_steps']
    grad_clip = train_config.get('grad_clip', 0.1)
    num_batches = len(loader)

    classification_losses = []
    localization_losses = []
    last_loss = 0.0

    for idx, (ims, targets, _) in enumerate(tqdm(loader, desc='Training')):
        targets = targets_to_device(targets, device)
        images = batch_images_to_device(ims, device)

        batch_losses = model(images, targets)['loss']

        loss = (sum(batch_losses['classification']) +
                sum(batch_losses['bbox_regression']))

        classification_losses.append(sum(batch_losses['classification']).item())
        localization_losses.append(sum(batch_losses['bbox_regression']).item())

        loss = loss / acc_steps
        last_loss = loss.detach()

        if not torch.isfinite(loss):
            raise RuntimeError(
                'Loss became non-finite at step {}. Exiting.'.format(steps))

        loss.backward()

        if ((idx + 1) % acc_steps == 0) or (idx + 1 == num_batches):
            if grad_clip:
                # Deformable sampling offsets are the usual source of a spike;
                # DETR-family training is normally clipped at 0.1.
                torch.nn.utils.clip_grad_norm_(
                    [p for p in model.parameters() if p.requires_grad],
                    grad_clip)
            optimizer.step()
            optimizer.zero_grad()

        if steps % train_config['log_steps'] == 0:
            print('Classification Loss : {:.4f} | Localization Loss : '
                  '{:.4f}'.format(np.mean(classification_losses),
                                  np.mean(localization_losses)),
                  scheduler.get_last_lr())

        steps += 1

    return (float(np.mean(classification_losses)),
            float(np.mean(localization_losses)),
            steps,
            last_loss)


@torch.no_grad()
def validate(model, loader, device):
    r"""
    Run the validation pass.

    ``model.eval()`` disables dropout and, as a side effect of how DETR.forward
    is written, also makes it emit detections alongside the losses. Only the
    losses are used here.

    :return: (mean classification loss, mean localization loss)
    """
    model.eval()
    classification_losses = []
    localization_losses = []

    for ims, targets, _ in tqdm(loader, desc='Validating'):
        targets = targets_to_device(targets, device)
        images = batch_images_to_device(ims, device)

        batch_losses = model(images, targets)['loss']

        classification_losses.append(sum(batch_losses['classification']).item())
        localization_losses.append(sum(batch_losses['bbox_regression']).item())

    return (float(np.mean(classification_losses)),
            float(np.mean(localization_losses)))


def train_rt_detr(config_path='rt_detr_config.yaml', resume=True):
    r"""
    Full RT-DETR training run on VOC, driven by the YAML config.

    Resumes from ``{task_name}/{ckpt_name}`` if it exists, then loops epochs,
    validating and checkpointing after each. Results are appended to
    ``{task_name}/train_results.txt`` so a killed-and-restarted run keeps its
    history. The best-validation epoch is kept separately as
    ``{task_name}/best_{ckpt_name}``.

    :param config_path: path to the YAML config
    :param resume: load the rolling checkpoint if one exists
    :return: the trained model
    """
    with open(config_path, 'r') as file:
        config = yaml.safe_load(file)
    dataset_config = config['dataset_params']
    model_config = config['model_params']
    train_config = config['train_params']
    print(config)

    set_seed(train_config['seed'])
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print('Device: {}'.format(device))

    # Seeded generator so the train/val split is identical across restarts -
    # otherwise a resumed run validates on images it has already trained on.
    generator = torch.Generator().manual_seed(train_config['seed'])
    train_loader, val_loader, _ = build_train_val_loaders(
        dataset_config, train_config, generator=generator)

    model = DETR(model_config,
                 num_classes=dataset_config['num_classes'],
                 bg_class_idx=dataset_config['bg_class_idx'])
    model.to(device)

    trainable = [p for p in model.parameters() if p.requires_grad]
    print('Trainable parameters: {:,} across {} tensors'.format(
        sum(p.numel() for p in trainable), len(trainable)))

    # Built before any resume, so restored state lands in the objects the loop
    # actually uses.
    optimizer = torch.optim.AdamW(params=trainable,
                                  lr=train_config['lr'],
                                  weight_decay=1E-4)
    scheduler = MultiStepLR(optimizer,
                            milestones=train_config['lr_steps'],
                            gamma=0.1)

    task_dir = train_config['task_name']
    os.makedirs(task_dir, exist_ok=True)
    result_path = os.path.join(task_dir, 'train_results.txt')
    ckpt_path = os.path.join(task_dir, train_config['ckpt_name'])
    best_ckpt_path = os.path.join(task_dir, 'best_' + train_config['ckpt_name'])

    # Read back so a restarted run cannot overwrite a better epoch with a worse one
    min_val_loss = read_min_val_loss(best_ckpt_path) if resume else float('inf')

    start_epoch, steps = 0, 0
    if resume and os.path.exists(ckpt_path):
        print('Loading checkpoint as one exists at {}'.format(ckpt_path))
        start_epoch, steps = load_checkpoint(ckpt_path, model, optimizer,
                                             scheduler, map_location=device)
        print('Resuming at epoch {} with lr {:g}'.format(
            start_epoch + 1, scheduler.get_last_lr()[0]))

    num_epochs = train_config['num_epochs']
    last_loss = 0.0

    for epoch in range(start_epoch, num_epochs):
        epoch_start = time.time()
        print('\n=== Epoch {}/{} ==='.format(epoch + 1, num_epochs))

        try:
            train_cls_loss, train_loc_loss, steps, last_loss = train_one_epoch(
                model, train_loader, optimizer, scheduler, device,
                train_config, steps=steps)
        except RuntimeError as exc:
            with open(result_path, 'a') as log:
                log.write('{}\n'.format(exc))
            raise

        print('Running Validation for Epoch {}...'.format(epoch + 1))
        val_cls_loss, val_loc_loss = validate(model, val_loader, device)

        scheduler.step()

        print('Finished epoch {} in {:.1f}s'.format(epoch + 1,
                                                    time.time() - epoch_start))
        print('Classification Loss : {:.4f} | Localization Loss : {:.4f}'.format(
            train_cls_loss, train_loc_loss))
        print('Val Class Loss: {:.4f} | Val Loc Loss: {:.4f}'.format(
            val_cls_loss, val_loc_loss))

        with open(result_path, 'a') as log:
            log.write(
                'Epoch {}: Train_Class_Loss={:.6f}, Train_Loc_Loss={:.6f}, '
                'Val_Class_Loss={:.6f}, Val_Loc_Loss={:.6f}\n'.format(
                    epoch + 1, train_cls_loss, train_loc_loss,
                    val_cls_loss, val_loc_loss))

        # epoch + 1 so a resumed run starts on the next epoch, not this one again
        save_checkpoint(model, optimizer, scheduler, epoch + 1, last_loss,
                        steps, ckpt_path)

        val_loss = val_cls_loss + val_loc_loss
        if val_loss < min_val_loss:
            min_val_loss = val_loss
            save_checkpoint(model, optimizer, scheduler, epoch + 1, last_loss,
                            steps, best_ckpt_path,
                            extra={'min_val_loss': min_val_loss,
                                   'train_cls_loss': train_cls_loss,
                                   'train_loc_loss': train_loc_loss,
                                   'val_cls_loss': val_cls_loss,
                                   'val_loc_loss': val_loc_loss})
            print('New best val loss {:.4f}, saved to {}'.format(
                min_val_loss, best_ckpt_path))

    with open(result_path, 'a') as log:
        log.write('Done Training...\n')
    print('Done Training...')
    return model


trained_model = train_rt_detr('rt_detr_config.yaml')